In [1]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25\
requests beautifulsoup4\
nltk\
reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from google.colab import userdata
from pageindex import PageIndexClient

import requests
import time
import csv
from bs4 import BeautifulSoup

import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

from rank_bm25 import BM25Okapi

from pydantic import Field
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

import numpy as np

In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))

WEB SCRAPING (did it again as dataset was v small in week 1 assignment)


In [4]:
# fetch html
url = "https://web.archive.org/web/20211101000000/https://www.bbc.com/news/world"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# parse html
soup = BeautifulSoup(response.text, 'html.parser')

# extract links
article_links = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/news/world-" in href and not href.endswith("/"):
        article_links.add(href)

article_links = list(article_links)[:100]

articles_corpus = []

for link in article_links:
    try:
      article_url = (
            link
            if link.startswith("http")
            else f"https://web.archive.org{link}"
      )
      response = requests.get(article_url, headers=headers)
      article_soup = BeautifulSoup(response.text, 'html.parser')

      title_tag = article_soup.find("h1")
      title = title_tag.text.strip() if title_tag else "No title found"

      para = article_soup.find_all("p")
      body_text = " ".join([p.text.strip() for p in para if len(p.text.strip()) > 20])

      if len(body_text.split()) < 100:
            continue

      articles_corpus.append({"title": title, "text": body_text})

      if len(articles_corpus) >= 100:
          break

      time.sleep(0.5)

    except:
      continue

filename = "raw_articles_corpus.csv"
with open(filename, mode = "w", newline = "", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["title", "text"])
    writer.writeheader()
    writer.writerows(articles_corpus)

In [5]:
for articles in articles_corpus:
    print(articles["title"])
    print(articles["text"])

Aukus: French president says Australian PM lied over submarine deal
French President Emmanuel Macron has said Australia's PM Scott Morrison lied to him about a scrapped submarine deal. Asked whether he thinks Mr Morrison was untruthful, the president replied: "I don't think, I know." Mr Macron was furious after Australia cancelled a $37bn (Â£27bn) deal to build 12 submarines, and instead negotiated a new defence pact with the US and the UK - the so-called Aukus. Mr Morrison denies that he was dishonest. The pair's meeting at the G20 summit was their first since the row erupted in September. On the sidelines of the gathering in Rome, President Macron was asked by an Australian journalist whether he could trust Mr Morrison again. "We will see what he will deliver," Mr Macron answered. "I have a lot of respect for your country. I have a lot of respect and a lot of friendship for your people. I just say when we have respect, you have to be true and you have to behave in line and consistent

In [6]:
df = pd.read_csv("raw_articles_corpus.csv")
print(len(df))

19


In [7]:
df.head()

,title,text
0,Aukus: French president says Australian PM lie...,French President Emmanuel Macron has said Aust...
1,FBI arrests suspected neo-Nazis ahead of Virgi...,The FBI has arrested three suspected members o...
2,Igor Kirillov: TV man known as the face of the...,"By Steve RosenbergBBC News, Moscow Igor Kirill..."
3,Climate change: The Indian village that could ...,"This video can not be played Uppada, a coastal..."
4,'Father of tiramisu' Ado Campeol dies aged 93,"Restaurateur Ado Campeol, dubbed ""the father o..."


PREPROCESSING

In [8]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = stopwords.words('english')
punctuation_marks = list(string.punctuation)

drop_list = stop_words + punctuation_marks

def clean_text(text):
    text = str(text).lower()
    words = word_tokenize(text)
    clean_words = []
    for word in words:
        if word not in drop_list:
            clean_words.append(word)

    return " ".join(clean_words)

df['cleaned_title'] = df['title'].apply(clean_text)
df['cleaned_text'] = df['text'].apply(clean_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
df.head()

,title,text,cleaned_title,cleaned_text
0,Aukus: French president says Australian PM lie...,French President Emmanuel Macron has said Aust...,aukus french president says australian pm lied...,french president emmanuel macron said australi...
1,FBI arrests suspected neo-Nazis ahead of Virgi...,The FBI has arrested three suspected members o...,fbi arrests suspected neo-nazis ahead virginia...,fbi arrested three suspected members neo-nazi ...
2,Igor Kirillov: TV man known as the face of the...,"By Steve RosenbergBBC News, Moscow Igor Kirill...",igor kirillov tv man known face ussr dies 89,steve rosenbergbbc news moscow igor kirillov m...
3,Climate change: The Indian village that could ...,"This video can not be played Uppada, a coastal...",climate change indian village could disappear ...,video played uppada coastal village southern i...
4,'Father of tiramisu' Ado Campeol dies aged 93,"Restaurateur Ado Campeol, dubbed ""the father o...",'father tiramisu ado campeol dies aged 93,restaurateur ado campeol dubbed `` father tira...


CONVERTING INTO PDF (for pageindex)

In [10]:
pdf = SimpleDocTemplate("corpus.pdf")
styles = getSampleStyleSheet()

components = []

for index, row in df.iterrows():
    components.append(Paragraph(str(row['cleaned_title']), styles["Heading2"]))
    components.append(Paragraph(str(row['cleaned_text']), styles["BodyText"]))
    components.append(PageBreak())

pdf.build(components)
print("PDF Created")



PDF Created


INDEXING

In [11]:
response = pi_client.submit_document("corpus.pdf")
doc_id = response["doc_id"]
print(doc_id)

while True:
    info = pi_client.get_document(doc_id)
    status = info["status"]
    print(status)

    if status == "completed":
        break

    time.sleep(10)

pi-cmpzj9142000z01qxxw348e2b
queued
completed


EXTRACTING NODES

In [12]:
tree = pi_client.get_tree(doc_id)

In [13]:
tree.keys()

dict_keys(['doc_id', 'status', 'retrieval_ready', 'result', 'metadata'])

LANGCHAIN DOCS

In [15]:
langchain_docs = []
for node in tree["result"]:
    doc = Document(
        page_content=str(node.get("text", "")),
        metadata={
            "title": node.get("title", ""),
            "node_id": node.get("node_id"),
            "parent": node.get("parent_id"),
            "level": node.get("level")
        }
    )
    langchain_docs.append(doc)

In [16]:
len(langchain_docs)

19

BM25 CORPUS

In [17]:
tokenized_corpus = [doc.page_content.lower().split() for doc in langchain_docs]
bm25 = BM25Okapi(tokenized_corpus)

RETRIEVAL

In [18]:
class BM25Retriever(BaseRetriever):
    bm25: BM25Okapi = Field(description="The underlying rank-bm25 index reference")
    documents: list = Field(description="List of formal LangChain document instances")
    k: int = 5

    def _get_relevant_documents(self, query: str):
        # Process the incoming question into matching text tokens
        query_tokens = query.lower().split()
        scores = self.bm25.get_scores(query_tokens)

        # Pull top-k indices with the highest score weights using NumPy sorting
        top_idx = np.argsort(scores)[::-1][:self.k]
        return [self.documents[i] for i in top_idx]

retriever = BM25Retriever(bm25=bm25, documents=langchain_docs, k=5)

TESTING

In [19]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

system_prompt = (
    "You are an assistant answering questions strictly based on the provided context.\n"
    "If the answer is not contained within the context, state 'Information not found.'\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [20]:
document_chain = create_stuff_documents_chain(llm, prompt_template)

retrieval_chain = create_retrieval_chain(retriever, document_chain)
print("RAG System Pipeline Live.")

RAG System Pipeline Live.


In [22]:
full_corpus_text = "\n\n".join([doc.page_content for doc in langchain_docs])

questions = [
    "Why was Fumio Kishida's Liberal Democratic Party being criticised before the election?",
    "What did French President Emmanuel Macron say when asked if Scott Morrison lied to him?",
    "Which web servers or targets were impacted by the Tokyo subway knife attack?",
    "What caused the fatal incident during the festival in the Spanish city of Onda?"
]

eval_results = []

for q in questions:
    print(f"\nEvaluating Question: '{q}'")

    rag_start = time.time()
    rag_answer = retrieval_chain.invoke({"input": q})["answer"]
    rag_latency = time.time() - rag_start

    # Track Baseline Full Context Speed
    naive_start = time.time()
    naive_response = llm.invoke(f"Context:\n{full_corpus_text}\n\nQuestion: {q}\nAnswer strictly based on context:")
    naive_answer = naive_response.content
    naive_latency = time.time() - naive_start

    time.sleep(20)

avg_rag = np.mean([r["rag_time"] for r in eval_results])
avg_naive = np.mean([r["naive_time"] for r in eval_results])

print("comparison")
print(f"Average Vectorless RAG Pipeline Latency: {avg_rag:.4f} seconds")
print(f"Average Naive Full Context Baseline Latency:  {avg_naive:.4f} seconds")
print(f"RAG Pipeline runs {avg_naive - avg_rag:.4f} seconds faster per request.")


Evaluating Question: 'Why was Fumio Kishida's Liberal Democratic Party being criticised before the election?'


ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (UNAUTHENTICATED): 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}}